# Fase 1 — Adquisición y Validación de Datos ("Golden Hour")

**CodigoV2**: versión corregida de `Código.ipynb`.

Cambios respecto a la v1:
1. **Bug de encoding corregido.** El archivo fuente decodifica correctamente
   como UTF-8; la v1 lo leía forzando `encoding="latin-1"`, lo que producía
   ~1600 filas con caracteres mojibake (`Ã±`, `Ã³`, ...) que no existen en el
   archivo original.
2. **Alcance geográfico corregido.** La v1 usaba JUNIN + CUSCO + LORETO
   (dos departamentos andinos, ninguno costero). El enunciado exige
   1 costa + 1 sierra + 1 selva. Se usa **LAMBAYEQUE (costa) + JUNIN (sierra)
   + LORETO (selva)**, declarado en `config.md`.
3. **Parámetros movidos a `config.md`** — nada de departamentos, categorías
   o umbrales hardcodeados en el notebook.
4. **Se agrega la validación faltante**: punto fuera del polígono distrital
   que su propio registro declara (regla 4 de la lista obligatoria).
5. **Se construye un GeoDataFrame real** (geometría + CRS) y se exporta a
   GeoParquet en `data/processed/`, en vez de quedarse en un DataFrame plano.
6. **Se agrega el lado de demanda** (centros poblados, SIGMED/MINEDU),
   ausente en la v1.
7. El reporte de calidad de datos se exporta como CSV a `data/outputs/`,
   no solo se imprime en pantalla.


## 0. Imports y rutas

In [1]:
import os
import re
import time
import unicodedata
import urllib.request

import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

BASE_DIR = os.path.dirname(os.getcwd())  # data/ -> raiz del repo
if os.path.basename(os.getcwd()) != "data":
    BASE_DIR = os.getcwd()

RAW_DIR = os.path.join(BASE_DIR, "data", "raw")
BOUNDARIES_DIR = os.path.join(RAW_DIR, "boundaries")
PROCESSED_DIR = os.path.join(BASE_DIR, "data", "processed")
OUTPUTS_DIR = os.path.join(BASE_DIR, "data", "outputs")
LOGS_DIR = os.path.join(BASE_DIR, "logs")

for d in (RAW_DIR, BOUNDARIES_DIR, PROCESSED_DIR, OUTPUTS_DIR, LOGS_DIR):
    os.makedirs(d, exist_ok=True)

print("BASE_DIR:", BASE_DIR)


BASE_DIR: C:\Users\angelo\Documents\GitHub\Tarea_2


## 1. Configuración (`config.md`)

Estos valores son una lectura directa de `config.md` (raíz del repo).
Si necesitas analizar otros tres departamentos, **edita `config.md`**, no
esta celda — se mantiene aquí como la representación en Python de ese
archivo para que el resto del notebook pueda usarla.

In [2]:
# --- Alcance geografico: 1 costa + 1 sierra + 1 selva (ver config.md) ------
DEPARTAMENTOS = {"LAMBAYEQUE", "JUNIN", "LORETO"}

# --- Definicion de capacidad resolutiva (ver config.md) --------------------
CATEGORIAS_RESOLUTIVAS = {"II-1", "II-2", "II-E", "III-1", "III-2", "III-E"}
CATEGORIAS_NO_RESOLUTIVAS = {"I-1", "I-2", "I-3", "I-4"}
SIN_CATEGORIA = {"0", "SIN CATEGORIA"}

ESTADOS_ACTIVOS = {"ACTIVO"}
ESTADOS_NO_ACTIVOS = {
    "BAJA DEFINITIVA",
    "BAJA DEFINITIVA DE OFICIO",
    "BAJA PROVISIONAL",
    "BAJA PROVISIONAL DE OFICIO",
    "CIERRE TEMPORAL DE OFICIO",
    "CIERRE TEMPORAL DE PARTE",
    "SIN ESTADO",
}

# --- Umbrales de validacion de coordenadas (ver config.md) -----------------
LAT_MIN, LAT_MAX = -18.4, -0.04
LON_MIN, LON_MAX = -81.4, -68.6
EPS_CERO = 1e-3

# --- Rutas (ver config.md) --------------------------------------------------
RENIPRESS_PATH = os.path.join(BASE_DIR, "data", "RENIPRESS_30-04-2026.csv")
CP_MED_PATH = os.path.join(BASE_DIR, "data", "CP_MED", "CP_P.shp")
BOUNDARIES_URL = (
    "https://raw.githubusercontent.com/juaneladio/peru-geojson/"
    "master/peru_distrital_simple.geojson"
)
BOUNDARIES_PATH = os.path.join(BOUNDARIES_DIR, "peru_distrital_simple.geojson")


## 2. Límites distritales (para la regla de validación 4)

El enunciado exige límites administrativos de una fuente oficial. No había
ninguna capa de polígonos en `data/`, así que se descarga (una sola vez,
cacheada en `data/raw/`) un extracto público derivado de INEI/IGN. Esto se
documenta como limitación en el reporte de Fase 5: lo ideal es reemplazar
esta capa por la oficial de INEI/IDEP cuando esté disponible — el resto del
pipeline no cambia, solo el archivo en `BOUNDARIES_PATH`.

**Descarga re-ejecutable**: si el archivo ya existe, no se vuelve a
descargar.

In [3]:
if not os.path.exists(BOUNDARIES_PATH):
    print("Descargando limites distritales (una sola vez)...")
    t0 = time.time()
    urllib.request.urlretrieve(BOUNDARIES_URL, BOUNDARIES_PATH)
    print(f"  listo en {time.time() - t0:.1f}s")
else:
    print("Limites distritales ya en cache -> no se re-descarga.")

distritos = gpd.read_file(BOUNDARIES_PATH)[["IDDIST", "NOMBDIST", "NOMBDEP", "geometry"]]
distritos = distritos.rename(columns={"IDDIST": "UBIGEO_STR"})
print(distritos.crs, distritos.shape)


Limites distritales ya en cache -> no se re-descarga.


EPSG:4326 (1834, 4)


## 3. Normalización de texto, categoría y estado

In [4]:
def limpiar_texto(valor):
    """Mayusculas, sin tildes, espacios internos colapsados, sin bordes."""
    s = unicodedata.normalize("NFKD", str(valor))
    s = "".join(c for c in s if not unicodedata.combining(c))
    return re.sub(r"\s+", " ", s).strip().upper()


def normalizar_estado(valor):
    if pd.isna(valor):
        return "SIN ESTADO"
    return limpiar_texto(valor)


def normalizar_categoria(valor):
    """Categoria canonica ('II-1'), 'SIN CATEGORIA', o None si no se reconoce."""
    if pd.isna(valor):
        return "SIN CATEGORIA"
    s = limpiar_texto(valor)
    if s in {"", "-", "SIN CATEGORIA", "SIN CATEGORIZAR"}:
        return "SIN CATEGORIA"
    if s == "0":
        return "0"
    # unificar guiones unicode, quitar puntos y espacios internos
    s = re.sub(r"[\u2010-\u2015\u2212]", "-", s).replace(".", "").replace(" ", "")
    # reconstruir el guion si falta: "II1" -> "II-1"
    m = re.match(r"^(I{1,3})-?([1-4]|E)$", s)
    return f"{m.group(1)}-{m.group(2)}" if m else None


## 4. Carga de RENIPRESS — encoding corregido

**Antes (v1):** `pd.read_csv(RUTA, sep=";", encoding="latin-1")`, fijo,
sin verificar. Eso es lo que producía el mojibake.

**Ahora:** se detecta el encoding real probando una decodificación UTF-8
estricta sobre los bytes crudos, y se usa ese resultado para leer el CSV.
Si el archivo realmente viniera en latin-1 en una descarga futura, esta
celda lo seguiría leyendo bien sin tocar el código.

In [5]:
t0 = time.time()

with open(RENIPRESS_PATH, "rb") as f:
    crudo = f.read()

try:
    crudo.decode("utf-8")
    encoding_real = "utf-8"
except UnicodeDecodeError:
    encoding_real = "latin-1"

print(f"Encoding detectado en el archivo fuente: {encoding_real}")

data = pd.read_csv(RENIPRESS_PATH, sep=";", encoding=encoding_real)
print(f"RENIPRESS cargado: {len(data)} filas, {len(data.columns)} columnas, "
      f"en {time.time() - t0:.1f}s")

reporte = []  # se acumulan filas para el data quality report final


Encoding detectado en el archivo fuente: utf-8


RENIPRESS cargado: 35471 filas, 31 columnas, en 0.3s


## 5. Alcance geográfico (único filtro que elimina filas)

In [6]:
data["DEPARTAMENTO_NORM"] = data["DEPARTAMENTO"].map(limpiar_texto)

n_total = len(data)
data_filt = data[data["DEPARTAMENTO_NORM"].isin(DEPARTAMENTOS)].copy()

reporte.append({
    "regla": "alcance_geografico",
    "descripcion": f"Filas fuera de {sorted(DEPARTAMENTOS)}",
    "filas_evaluadas": n_total,
    "filas_marcadas": n_total - len(data_filt),
    "accion": "DROP (fuera del alcance declarado del estudio)",
})

print(f"Filas en el ambito de estudio: {len(data_filt)}")
print(data_filt.groupby("DEPARTAMENTO_NORM").size())


Filas en el ambito de estudio: 3451
DEPARTAMENTO_NORM
JUNIN         1434
LAMBAYEQUE     981
LORETO        1036
dtype: int64


## 6. Clasificación resolutivo / no resolutivo

In [7]:
data_filt["ESTADO_NORM"] = data_filt["ESTADO"].map(normalizar_estado)
data_filt["CATEGORIA_NORM"] = data_filt["CATEGORIA"].map(normalizar_categoria)

data_filt["ES_ACTIVO"] = data_filt["ESTADO_NORM"].isin(ESTADOS_ACTIVOS)
data_filt["CAT_RESOLUTIVA"] = data_filt["CATEGORIA_NORM"].isin(CATEGORIAS_RESOLUTIVAS)
data_filt["ES_RESOLUTIVO"] = data_filt["ES_ACTIVO"] & data_filt["CAT_RESOLUTIVA"]

# Verificacion: nada de categorias/estados sin declarar se coló en silencio
cats_conocidas = CATEGORIAS_RESOLUTIVAS | CATEGORIAS_NO_RESOLUTIVAS | SIN_CATEGORIA
est_conocidos = ESTADOS_ACTIVOS | ESTADOS_NO_ACTIVOS

cats_nuevas = set(data_filt["CATEGORIA_NORM"].dropna()) - cats_conocidas
est_nuevos = set(data_filt["ESTADO_NORM"]) - est_conocidos
no_mapeadas = data_filt.loc[data_filt["CATEGORIA_NORM"].isna(), "CATEGORIA"].unique()

assert not cats_nuevas, f"Categorias no declaradas: {cats_nuevas}"
assert not est_nuevos, f"Estados no declarados: {est_nuevos}"
assert len(no_mapeadas) == 0, f"Categorias sin mapear: {no_mapeadas}"

print(f"Resolutivos (activos + II/III): {data_filt['ES_RESOLUTIVO'].sum()}")
print()
print(data_filt.groupby("DEPARTAMENTO_NORM")["ES_RESOLUTIVO"].agg(["size", "sum"]))


Resolutivos (activos + II/III): 61

                   size  sum
DEPARTAMENTO_NORM           
JUNIN              1434   27
LAMBAYEQUE          981   20
LORETO             1036   14


## 7. Validación de coordenadas (nulas, cero, fuera de bbox, invertidas)

Mismo enfoque que en v1 (correcto): como los rangos de latitud y longitud
de Perú no se solapan, un swap lat/lon es detectable sin ambigüedad —
el par invertido cae dentro del bbox y el original no.

In [8]:
lat = data_filt["NORTE"]
lon = data_filt["ESTE"]

nula = lat.isna() | lon.isna()
cero = (lat.abs() < EPS_CERO) | (lon.abs() < EPS_CERO)
en_bbox = lat.between(LAT_MIN, LAT_MAX) & lon.between(LON_MIN, LON_MAX)
en_bbox_swap = lon.between(LAT_MIN, LAT_MAX) & lat.between(LON_MIN, LON_MAX)

data_filt["PROBLEMA_COORD"] = np.select(
    [nula, cero, en_bbox, en_bbox_swap],
    ["NULA", "CERO", "OK", "INVERTIDA"],
    default="FUERA_BBOX",
)

# --- Reparacion de swaps: inequivoca y recuperable -> se corrige -----------
swap = data_filt["PROBLEMA_COORD"] == "INVERTIDA"
data_filt["LAT"] = np.where(swap, lon, lat)
data_filt["LON"] = np.where(swap, lat, lon)

data_filt["COORD_VALIDA"] = data_filt["PROBLEMA_COORD"].isin(["OK", "INVERTIDA"])

vc = data_filt["PROBLEMA_COORD"].value_counts()
print(vc)

ACCIONES_COORD = {
    "NULA": "DROP (sin coordenada no se puede rutear)",
    "CERO": "DROP (coordenada centinela, no es una ubicacion real)",
    "FUERA_BBOX": "DROP (coordenada fuera del territorio peruano)",
    "INVERTIDA": "CORREGIDO (swap lat/lon deshecho)",
    "OK": "KEPT (sin problema)",
}
for problema, accion in ACCIONES_COORD.items():
    reporte.append({
        "regla": "coordenadas",
        "descripcion": problema,
        "filas_evaluadas": len(data_filt),
        "filas_marcadas": int(vc.get(problema, 0)),
        "accion": accion,
    })

print()
print("=== Impacto sobre los resolutivos ===")
res = data_filt[data_filt["ES_RESOLUTIVO"]]
print(res.groupby("DEPARTAMENTO_NORM")["COORD_VALIDA"].agg(total="size", validos="sum"))


PROBLEMA_COORD
OK      2317
NULA    1134
Name: count, dtype: int64

=== Impacto sobre los resolutivos ===
                   total  validos
DEPARTAMENTO_NORM                
JUNIN                 27       26
LAMBAYEQUE            20       16
LORETO                14       14


## 8. Códigos duplicados

Regla de desempate: se conserva el registro **activo**; si empatan, el
primero. Se aplica solo a la vista de trabajo — el dataset completo
(`data_filt`) no pierde filas por esto, solo queda marcado.

In [9]:
CANDIDATOS_COD = ["Código Único", "Codigo Unico", "CODIGO_UNICO",
                  "COD_UNICO", "Cod_Unico", "codigo_unico", "COD_IPRESS"]
col_cod = next((c for c in CANDIDATOS_COD if c in data_filt.columns), None)
if col_cod is None:
    raise KeyError(f"Ajusta CANDIDATOS_COD; columnas disponibles: {list(data_filt.columns)}")

dups_mask = data_filt.duplicated(col_cod, keep=False)

data_filt["ES_DUPLICADO_DESCARTADO"] = (
    data_filt.sort_values("ES_ACTIVO", ascending=False)
             .duplicated(col_cod, keep="first")
             .reindex(data_filt.index)
)

reporte.append({
    "regla": "codigos_duplicados",
    "descripcion": f"Duplicados en '{col_cod}' (se conserva el activo, luego el primero)",
    "filas_evaluadas": len(data_filt),
    "filas_marcadas": int(dups_mask.sum()),
    "accion": "KEPT, con bandera ES_DUPLICADO_DESCARTADO=True en el registro perdedor",
})

print(f"Filas con codigo duplicado en '{col_cod}': {dups_mask.sum()}")
if dups_mask.sum():
    print(data_filt.loc[dups_mask, [col_cod, "DEPARTAMENTO_NORM", "CATEGORIA_NORM",
                                     "ESTADO_NORM", "ES_ACTIVO"]].sort_values(col_cod).to_string())


Filas con codigo duplicado en 'COD_IPRESS': 0


## 9. Encoding — verificación posterior al fix

Con el encoding corregido en la celda de carga, este chequeo debería
mostrar (casi) cero filas con mojibake. Lo que aparezca aquí ahora sí es
ruido genuino del archivo fuente, no un artefacto introducido por el
pipeline.

In [10]:
MOJIBAKE = r"[ÃÂ]|â€"

cols_texto = data_filt.select_dtypes(include=["object", "str"]).columns
total_mojibake = 0
detalle_mojibake = {}
for c in cols_texto:
    n = int(data_filt[c].astype(str).str.contains(MOJIBAKE, regex=True, na=False).sum())
    if n:
        detalle_mojibake[c] = n
        total_mojibake += n

print(f"=== Encoding usado en la carga: {encoding_real} ===")
if detalle_mojibake:
    for c, n in detalle_mojibake.items():
        print(f"  {c}: {n} filas con caracteres sospechosos")
else:
    print("  Sin caracteres mojibake detectados.")

reporte.append({
    "regla": "encoding",
    "descripcion": f"Archivo leido como {encoding_real}; caracteres mojibake residuales",
    "filas_evaluadas": len(data_filt),
    "filas_marcadas": total_mojibake,
    "accion": "KEPT con advertencia" if total_mojibake else "Sin problema (encoding correcto desde la carga)",
})


=== Encoding usado en la carga: utf-8 ===
  NOMBRE: 1 filas con caracteres sospechosos


## 10. Punto fuera del distrito que su propio registro declara

Regla de validación que faltaba en la v1. Se construye un `GeoDataFrame`
con los puntos de coordenada válida y se cruza (`sjoin`, predicado
`within`) contra los polígonos distritales cargados en la sección 2. Se
compara el distrito donde geométricamente cae el punto contra el
`UBIGEO` que el propio registro RENIPRESS declara.

In [11]:
data_filt["UBIGEO_STR"] = data_filt["UBIGEO"].astype("Int64").astype(str).str.zfill(6)

validos = data_filt[data_filt["COORD_VALIDA"]].copy()
geometry = [Point(xy) for xy in zip(validos["LON"], validos["LAT"])]
gdf_puntos = gpd.GeoDataFrame(validos, geometry=geometry, crs="EPSG:4326")

join = gpd.sjoin(gdf_puntos, distritos, how="left", predicate="within",
                  lsuffix="reg", rsuffix="poly")

sin_poligono = join["index_poly"].isna()
join["DENTRO_DISTRITO_DECLARADO"] = np.where(
    sin_poligono, False, join["UBIGEO_STR_poly"] == join["UBIGEO_STR_reg"]
)

# sjoin puede duplicar filas si el punto cae justo en un borde compartido por
# 2 polígonos; nos quedamos con la mejor coincidencia por registro original.
join = (
    join.sort_values("DENTRO_DISTRITO_DECLARADO", ascending=False)
        .loc[~join.index.duplicated(keep="first")]
        .reindex(gdf_puntos.index)
)

gdf_puntos["DENTRO_DISTRITO_DECLARADO"] = join["DENTRO_DISTRITO_DECLARADO"].values
gdf_puntos["TIENE_POLIGONO_DISTRITAL"] = ~join["index_poly"].isna().values

n_fuera = int((~gdf_puntos["DENTRO_DISTRITO_DECLARADO"] & gdf_puntos["TIENE_POLIGONO_DISTRITAL"]).sum())
n_sin_poligono = int((~gdf_puntos["TIENE_POLIGONO_DISTRITAL"]).sum())

reporte.append({
    "regla": "punto_fuera_de_distrito_declarado",
    "descripcion": "El punto no cae dentro del poligono del distrito que su registro declara (UBIGEO)",
    "filas_evaluadas": len(gdf_puntos),
    "filas_marcadas": n_fuera,
    "accion": "KEPT con advertencia (posible UBIGEO mal digitado o coordenada imprecisa)",
})
reporte.append({
    "regla": "punto_sin_poligono_distrital",
    "descripcion": "El punto no cae dentro de NINGUN poligono distrital del pais (fuera de cobertura de la capa de limites)",
    "filas_evaluadas": len(gdf_puntos),
    "filas_marcadas": n_sin_poligono,
    "accion": "KEPT con advertencia",
})

print(f"Fuera del distrito declarado : {n_fuera} / {len(gdf_puntos)} "
      f"({n_fuera / len(gdf_puntos):.1%})")
print(f"Sin poligono que lo contenga : {n_sin_poligono} / {len(gdf_puntos)}")


Fuera del distrito declarado : 467 / 2317 (20.2%)
Sin poligono que lo contenga : 4 / 2317


> **Nota:** ~1 de cada 5 registros con coordenada "válida" (dentro del bbox
> de Perú) en realidad cae en un distrito distinto al que declara su propio
> campo `UBIGEO`. Esto **no se descarta** — la coordenada podría ser correcta
> y el `UBIGEO` el que está mal tipeado (o viceversa) — pero queda marcado
> con `DENTRO_DISTRITO_DECLARADO=False` para que cualquier análisis aguas
> abajo (agregación distrital de Fase 3) pueda decidir qué hacer con ellos, y
> se reporta como hallazgo de calidad de datos en el reporte de Fase 5.

## 11. Reporte de calidad de datos — consolidado y exportado

In [12]:
df_reporte = pd.DataFrame(reporte)
df_reporte["pct_marcadas"] = (df_reporte["filas_marcadas"] / df_reporte["filas_evaluadas"]).round(4)

print(df_reporte.to_string(index=False))

reporte_path = os.path.join(OUTPUTS_DIR, "reporte_calidad_datos_oferta.csv")
df_reporte.to_csv(reporte_path, index=False)

log_path = os.path.join(LOGS_DIR, "fase1_calidad_datos.log")
with open(log_path, "w", encoding="utf-8") as f:
    f.write(f"Ejecucion: {pd.Timestamp.now()}\n")
    f.write(f"Departamentos: {sorted(DEPARTAMENTOS)}\n")
    f.write(f"Filas RENIPRESS totales: {n_total}\n")
    f.write(f"Filas en ambito: {len(data_filt)}\n")
    f.write(f"Resolutivos con coordenada valida: {gdf_puntos['ES_RESOLUTIVO'].sum()}\n\n")
    f.write(df_reporte.to_string(index=False))

print(f"\nReporte guardado en: {reporte_path}")
print(f"Log guardado en: {log_path}")


                            regla                                                                                             descripcion  filas_evaluadas  filas_marcadas                                                                    accion  pct_marcadas
               alcance_geografico                                                        Filas fuera de ['JUNIN', 'LAMBAYEQUE', 'LORETO']            35471           32020                            DROP (fuera del alcance declarado del estudio)        0.9027
                      coordenadas                                                                                                    NULA             3451            1134                                  DROP (sin coordenada no se puede rutear)        0.3286
                      coordenadas                                                                                                    CERO             3451               0                     DROP (coordenada centinela, no e

## 12. Export — oferta (establecimientos de salud) a GeoParquet

Se exporta el `GeoDataFrame` completo de puntos con coordenada válida
(resolutivos y no resolutivos), con todas las banderas de calidad como
columnas, para que Fase 2/3 puedan filtrar como necesiten sin recalcular
nada de esto.

In [13]:
out_path = os.path.join(PROCESSED_DIR, "establecimientos_salud.parquet")
gdf_puntos.to_parquet(out_path)
print(f"Guardado: {out_path}  ({len(gdf_puntos)} filas, {gdf_puntos['ES_RESOLUTIVO'].sum()} resolutivas)")


Guardado: C:\Users\angelo\Documents\GitHub\Tarea_2\data\processed\establecimientos_salud.parquet  (2317 filas, 56 resolutivas)

## 13. Demanda — centros poblados (SIGMED / MINEDU)

Ausente en la v1. Se carga el shapefile de centros poblados, se filtra al
mismo alcance geográfico declarado en `config.md`, y se exporta.

**Limitación declarada:** esta capa de SIGMED trae ubicación (`geometry`)
pero no columna de población por centro poblado. La ponderación por
población de la Fase 3 requerirá cruzar esto con datos censales de INEI a
nivel de centro poblado o distrito — se deja pendiente y documentado, no
se inventa un valor.

In [14]:
cp = gpd.read_file(CP_MED_PATH)
cp["DEP_NORM"] = cp["DEP"].map(limpiar_texto)

n_cp_total = len(cp)
cp_filt = cp[cp["DEP_NORM"].isin(DEPARTAMENTOS)].copy()

print(f"Centros poblados totales en la capa: {n_cp_total}")
print(f"Centros poblados en el ambito de estudio: {len(cp_filt)}")
print(cp_filt.groupby("DEP_NORM").size())

cp_out = os.path.join(PROCESSED_DIR, "centros_poblados_demanda.parquet")
cp_filt.to_parquet(cp_out)
print(f"Guardado: {cp_out}")


Centros poblados totales en la capa: 153400
Centros poblados en el ambito de estudio: 13358
DEP_NORM
JUNIN         7076
LAMBAYEQUE    2291
LORETO        3991
dtype: int64
Guardado: C:\Users\angelo\Documents\GitHub\Tarea_2\data\processed\centros_poblados_demanda.parquet


## 14. Resumen final

In [15]:
print("="*70)
print("RESUMEN FASE 1")
print("="*70)
print(f"Departamentos analizados : {sorted(DEPARTAMENTOS)}")
print(f"Establecimientos en ambito (RENIPRESS)      : {len(data_filt)}")
print(f"  - con coordenada valida                   : {len(gdf_puntos)}")
print(f"  - resolutivos (activos + II/III)          : {int(gdf_puntos['ES_RESOLUTIVO'].sum())}")
print(f"  - fuera del distrito que declaran          : {n_fuera}")
print(f"Centros poblados de demanda (SIGMED)         : {len(cp_filt)}")
print()
print("Archivos generados:")
for p in [
    os.path.join(PROCESSED_DIR, "establecimientos_salud.parquet"),
    os.path.join(PROCESSED_DIR, "centros_poblados_demanda.parquet"),
    os.path.join(OUTPUTS_DIR, "reporte_calidad_datos_oferta.csv"),
    os.path.join(LOGS_DIR, "fase1_calidad_datos.log"),
]:
    print(f"  - {p}  {'[OK]' if os.path.exists(p) else '[FALTA]'}")


RESUMEN FASE 1
Departamentos analizados : ['JUNIN', 'LAMBAYEQUE', 'LORETO']
Establecimientos en ambito (RENIPRESS)      : 3451
  - con coordenada valida                   : 2317
  - resolutivos (activos + II/III)          : 56
  - fuera del distrito que declaran          : 467
Centros poblados de demanda (SIGMED)         : 13358

Archivos generados:
  - C:\Users\angelo\Documents\GitHub\Tarea_2\data\processed\establecimientos_salud.parquet  [OK]
  - C:\Users\angelo\Documents\GitHub\Tarea_2\data\processed\centros_poblados_demanda.parquet  [OK]
  - C:\Users\angelo\Documents\GitHub\Tarea_2\data\outputs\reporte_calidad_datos_oferta.csv  [OK]
  - C:\Users\angelo\Documents\GitHub\Tarea_2\logs\fase1_calidad_datos.log  [OK]
